# Experiment Benefit of Depth
In this notebook the benefit of depth is tested on a CNN base model that was established in the notebook "CNN-architecture-testing".

### Resaerch Question:
How does the depth of the neural network influence the model accuracy and loss?

### Expectation:
It is expected that the model accuracy will increase and the model loss will decrease respectively.\
But also the training time is expected to increase with more layers.\
It is thought that adding more hidden layers will have stagnating improvement untill to the point where there is no more improvement.

### Framework:
CNN with 

In [ ]:
# shallow_model class was here

In [ ]:
from src.cnn.cnn_models import DepthCNN
from src.cnn.cnn_utils import get_dataset, get_num_parameters, evaluate

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

In [ ]:
import copy
import time
from dataclasses import dataclass, asdict
from typing import Dict, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import wandb


# =========================================================
# CONFIG
# =========================================================

# class Config was here


# =========================================================
# MODEL
# =========================================================

# ConvBlock class was here
# DepthCNN class was here


# =========================================================
# UTILITIES
# =========================================================

# get_num_parameters was here



# =========================================================
# TRAIN / EVAL FUNCTIONS
# =========================================================




# evaluation() was here

# =========================================================
# TRAIN ONE MODEL WITH W&B
# =========================================================

# def train_model was here


# =========================================================
# TRAIN ALL DEPTH MODELS
# =========================================================

# Function to train models for all specified depths in the configuration, storing the trained models, their training histories, and final results in dictionaries for easy access and comparison.
def train_all_depth_models(
    train_dataset,
    val_dataset,
    cfg: Config
):
    all_models = {}
    all_histories = {}
    all_results = {}

    for depth in cfg.depths:
        model, history, result = train_model(
            depth=depth,
            train_data=train_dataset,
            val_dataset=val_dataset,
            cfg=cfg
        )

        all_models[depth] = copy.deepcopy(model).cpu() # store a copy of the trained model on CPU to save GPU memory, can be moved back to GPU for evaluation if needed
        all_histories[depth] = history
        all_results[depth] = result

    return all_models, all_histories, all_results


# evaluate_test_set was here


# =========================================================
# MAIN EXECUTION
# =========================================================

 #Defin configuration for training
cfg = Config(
    in_channels=3, # rgb
    num_classes=10,
    depths=(1, 2, 4, 8, 16, 32),
    batch_size=64,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-4,
    base_channels=32,
    max_channels=256,
    dropout=0.0, # set to zero for Dropout in ConvBlocks since it was found to not help in this setting and just increases training time, but can be easily tuned if needed
    do=0.5, # dropout rate for the final classifier head, can be tuned as a hyperparameter if needed
    num_workers=0,
    pin_memory=True,
    use_wandb=True,
    wandb_project="MPW-CNN",
    wandb_entity="MSE_DeLearn_SPR26",   # set to None if not needed
    wandb_mode="online",                # "offline" if online mode fails
    wandb_watch=False,                  # safer on Windows / VS Code notebooks
)

# train models for all specified depths and collect their results
all_models, all_histories, all_results = train_all_depth_models(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    cfg=cfg
)

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

for depth, result in all_results.items():
    print(
        f"Depth {depth:>2} | "
        f"params={result['num_parameters']:,} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_train_acc={result['best_train_acc']:.4f} | "
        f"best_train_loss={result['best_train_loss']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"final_train_acc={result['final_train_acc']:.4f} | "
        f"final_val_acc={result['final_val_acc']:.4f} | "
        f"avg_epoch_time={result['avg_epoch_time_sec']:.2f}s"
    )

# identify the best model based on validation accuracy and report its performance, optionally evaluate on the test set if available
best_depth = max(all_results.keys(), key=lambda d: all_results[d]["best_val_acc"])
best_model = all_models[best_depth]

print("\nBest model based on validation accuracy:")
print(f"Depth = {best_depth}")
print(f"Best validation accuracy = {all_results[best_depth]['best_val_acc']:.4f}")

# Optional test evaluation
# test_metrics = evaluate_test_set(best_model, test_dataset, cfg)
# print("\nTest set performance:")
# print(test_metrics)

# Optional: log summary of best model as separate W&B run --> stores a summary table comparing all depths and highlights the best one, useful for easy comparison and reporting in W&B dashboard
if cfg.use_wandb and cfg.wandb_mode != "disabled":
    run = wandb.init(
        project=cfg.wandb_project,
        entity=cfg.wandb_entity,
        mode=cfg.wandb_mode,
        name="cnn_depth_comparison_summary",
        reinit=True,
        settings=wandb.Settings(init_timeout=300),
    )

    comparison_table = wandb.Table(columns=[
        "depth",
        "num_parameters",
        "best_val_acc",
        "best_val_loss",
        "best_epoch",
        "final_train_acc",
        "final_val_acc",
        "avg_epoch_time_sec",
    ])

    for depth, result in all_results.items():
        comparison_table.add_data(
            result["depth"],
            result["num_parameters"],
            result["best_val_acc"],
            result["best_val_loss"],
            result["best_epoch"],
            result["best_train_acc"],
            result["best_train_loss"],
            result["final_train_acc"],
            result["final_val_acc"],
            result["avg_epoch_time_sec"],
        )

    wandb.log({
        "depth_comparison_table": comparison_table,
        "best_depth": best_depth,
        "best_depth_val_acc": all_results[best_depth]["best_val_acc"],
    })

    wandb.summary["best_depth"] = best_depth
    wandb.summary["best_depth_val_acc"] = all_results[best_depth]["best_val_acc"]
    wandb.finish()